# Bures-Wasserstein distance between geometries (paper Fig. 6, appendix)

* **Goal**: plot Fig. 6, the Bures-Wasserstein distance `d_BW` between the reconstructed geometries of two models or two areas, as a function of the number `k` of leading eigenvectors kept. It is the check that the conclusions of Fig. 4D-E, based on the subspace alignment, do not depend on the similarity measure: the alignment compares only the orientation of the leading eigenvectors, while `d_BW` also penalises differences in scale.
* **Distance**: for two metrics `A` and `B` with rank-`k` reconstructions `V_A diag(l_A) V_A^T` and `V_B diag(l_B) V_B^T`, `d_BW(A, B)^2 = tr(A) + tr(B) - 2 tr((A^(1/2) B A^(1/2))^(1/2))`. It is computed without forming the 4096 x 4096 matrices, for `k = 1..50` and for every pair of (encoder, area) configurations, and averaged over the test images that the two configurations share.
* **Four comparisons**: V1 against V4 within Inception and within ResNet-50 (same model, different areas, dashed), and Inception against ResNet-50 within V1 and within V4 (same area, different models, solid). Left: the multi-scale metric G(x). Right: the Fisher metric J(x).
* **What the paper reports**: for G(x) the cross-model distances (solid) lie below the cross-area distances (dashed); for J(x) the order reverses. This is the same order as in Fig. 4D-E.
* All supporting functions are in the `multigeom` package (`multigeom.geometry.metrics`, `multigeom.pipeline.alignment` and `multigeom.viz`); this notebook only sets parameters and calls them.

## 1. Inputs

The distances are computed from the per-image eigenvectors and eigenvalues of G(x) and J(x) in `runs/per_image/`, which notebook 2 (`2_subspace_alignment.ipynb`) estimates for the images and configurations you choose. The paper averages over 200 test images and all configurations; with fewer images the curves are noisier. At least two configurations are needed, and all four comparisons of the figure need inception and resnet50 for both V1 and V4.

In [ ]:
%%time

# import built-in packages
import os, sys
from pathlib import Path
import importlib

# move to the repository root (the folder containing environment.yml): data/ and runs/ are read from there
root = Path.cwd().resolve()
while not (root / "environment.yml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

# import custom packages
from multigeom import viz
from multigeom.pipeline import alignment as bw
importlib.reload(bw)      # update the module without restarting the kernel

# parameters
RUNS_DIR = "runs"         # folder with per_image/<encoder>/<area>/img*.pt (written by notebook 2)

summary = bw.print_inputs(RUNS_DIR)

## 2. The distances

Runs `multigeom.pipeline.alignment` (minutes at most, on the CPU) and writes `runs/metrics/cross_encoder_metrics.json`, which also holds the subspace alignment of Fig. 4D-E. It does nothing if fewer than two configurations have per-image files.

In [ ]:
%%time

metrics_path = bw.compute_metrics(RUNS_DIR, out_dir=f"{RUNS_DIR}/metrics")

## 3. Does the order match the paper?

The mean of `d_BW` over `k` for the two cross-model comparisons and the two cross-area comparisons, for each metric. The paper reports cross-model below cross-area for G(x) and the reverse for J(x).

In [ ]:
if metrics_path is None:
    print("Nothing to check yet: the alignment needs at least two configurations (see the message above).")
else:
    metrics = bw.load_metrics(metrics_path)
    order = bw.print_ordering(metrics)

## 4. Plot Fig. 6

Solid: different models, same area. Dashed: the same model, V1 versus V4. Bands: s.e.m. across the images. The paper also compares ON-OFF, for which there are no checkpoints here.

In [ ]:
%%time

if metrics_path is None:
    print("Nothing to plot yet: run the cells above first.")
else:
    viz.show_figure(viz.plot_bures_wasserstein(metrics))